# Keras fraud classifier

Run in a fresh environment with the TensorFlow requirements. The public repository needs no GitHub token.


## Execution provenance

All code cells below were executed on 16 November 2026 in an isolated CPU checkout. These are fresh verification-training outputs, recorded in `results/verification/2026-11-16/keras.json`. The published report and slides use the earlier frozen `results/keras.json`; the rerun did not overwrite that experiment. Different software, hardware and training randomness can change the metrics. See `docs/demo_readiness.md` for the comparison. Run on CPU with GPUs hidden (CUDA_VISIBLE_DEVICES=-1) and TensorFlow C++ logging set to level 3.


## Setup and shared data

In [1]:
from pathlib import Path
import os, subprocess
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (repo_root / "src" / "data.py").exists():
    repo_root = Path("/content/credit-card-fraud-detection")
    if not (repo_root / "src" / "data.py").exists():
        subprocess.run(["git", "clone", "https://github.com/VanshSharmaPES/credit-card-fraud-detection.git", str(repo_root)], check=True)
os.chdir(repo_root)
print("Project root:", Path.cwd())


Project root: /home/vanshsharma/ccfd-rerun/train


In [2]:
%pip install -q -r requirements/base.txt
%pip install -q -r requirements/keras.txt
import subprocess, sys
subprocess.run([sys.executable, "src/data.py"], check=True)
# In the full team repository, run the existing setup/data cells first.
# The team's src/data.py must expose load_splits() as agreed in the project plan.
from pathlib import Path
import sys, random, time, platform
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd() / "src"))
from data import load_splits
from evaluate import evaluate_model, permutation_importance
SEED = 42
random.seed(SEED); np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)
(X_train, y_train, X_val, y_val, X_test, y_test,
 row_id_val, row_id_test, amount_val, amount_test) = load_splits()
feature_names = list(X_train.columns)
X_train = np.asarray(X_train, dtype=np.float32)
X_val = np.asarray(X_val, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.int64)
y_val = np.asarray(y_val, dtype=np.int64)
y_test = np.asarray(y_test, dtype=np.int64)
class_weight = {0: 1.0, 1: float((y_train == 0).sum() / (y_train == 1).sum())}


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Raw rows: 284807, raw frauds: 492
Duplicates dropped: 1081
Rows after cleaning: 283726, frauds: 473
train rows:  170235  frauds:  284  fraud share: 0.1668%
val   rows:   56745  frauds:   94  fraud share: 0.1657%
test  rows:   56746  frauds:   95  fraud share: 0.1674%
Class weight (train normal / train fraud): 169951 / 284 = 598.4190


I0000 00:00:1794884883.961801   63046 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1794884886.202635   63046 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Train and evaluate

In [3]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation="relu"), layers.Dropout(0.2),
    layers.Dense(32, activation="relu"), layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(curve="PR", name="pr_auc")],
)
callbacks = [keras.callbacks.EarlyStopping(
    monitor="val_pr_auc", mode="max", patience=5, restore_best_weights=True
)]
start = time.perf_counter()
history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val), class_weight=class_weight,
    batch_size=2048, epochs=30, callbacks=callbacks, verbose=1,
)
train_seconds = time.perf_counter() - start
p_val = model.predict(X_val, batch_size=8192, verbose=0).ravel()
start = time.perf_counter()
p_test = model.predict(X_test, batch_size=8192, verbose=0).ravel()
predict_seconds = time.perf_counter() - start
Path("models").mkdir(exist_ok=True)
model.save("models/keras.keras")
result = evaluate_model(
    "keras", y_val, p_val, y_test, p_test, row_id_val, row_id_test,
    train_seconds, predict_seconds,
    notes="Dense(64,32) ReLU network; dropout 0.2; class weighted; early stopped on validation PR-AUC.",
    hardware=f"{platform.machine()} CPU",
)
result


Epoch 1/30


 1/84 ━━━━━━━━━━━━━━━━━━━━ 3:01 2s/step - loss: 3.7801 - pr_auc: 0.0415

13/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.2740 - pr_auc: 0.1410 

25/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.0734 - pr_auc: 0.2243

37/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.0019 - pr_auc: 0.3627

50/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.8900 - pr_auc: 0.4175

... (64 outputs left out)

72/84 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2068 - pr_auc: 0.6919

79/84 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2149 - pr_auc: 0.6830

84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.2109 - pr_auc: 0.6760 - val_loss: 0.0819 - val_pr_auc: 0.6632


{'framework': 'keras',
 'threshold': 0.9603972434997559,
 'precision': 0.8043478260869565,
 'recall': 0.7789473684210526,
 'f1': 0.7914438502673797,
 'pr_auc': 0.7380379708361576,
 'roc_auc': 0.9621689216244615,
 'tn': 56633,
 'fp': 18,
 'fn': 21,
 'tp': 74,
 'rmse': 0.17924741860922558,
 'mae': 0.12821787892635034,
 'r2': -18.22406125986268,
 'train_seconds': 6.847280861999934,
 'predict_seconds': 0.14708183800007646,
 'hardware': 'x86_64 CPU',
 'notes': 'Dense(64,32) ReLU network; dropout 0.2; class weighted; early stopped on validation PR-AUC.'}

## Validation permutation importance

Run this as an optional transparency output. It uses validation data and average precision; it does not change the trained model.

In [4]:
def keras_predict(frame):
    arr = np.asarray(frame, dtype=np.float32)
    return model.predict(arr, batch_size=8192, verbose=0).ravel()
perm = permutation_importance(keras_predict, pd.DataFrame(X_val, columns=feature_names), y_val, n_repeats=3, random_state=SEED)
Path("results").mkdir(exist_ok=True)
perm.to_csv("results/perm_keras.csv", index=False)
perm.head(10)

,feature,importance_mean,importance_std,baseline_pr_auc
0,V14,0.026686,0.002807,0.689137
1,V5,0.013928,0.000812,0.689137
2,V8,0.010580,0.006717,0.689137
3,V17,0.005279,0.001259,0.689137
4,V16,0.004661,0.003059,0.689137
5,V27,0.004411,0.003341,0.689137
6,V28,0.003623,0.003654,0.689137
7,V19,0.002713,0.001806,0.689137
8,V15,0.002272,0.005870,0.689137
9,Hour,0.001611,0.001141,0.689137
